In [1]:
%%writefile matrix_mul.cu
#include <iostream>
using namespace std;

// CUDA Kernel
__global__ void matrixMul(int *A, int *B, int *C, int m, int n, int p) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < m && col < p) {
        int sum = 0;

        for (int k = 0; k < n; k++) {
            sum += A[row * n + k] * B[k * p + col];
        }

        C[row * p + col] = sum;
    }
}

int main() {
    int m, n, p;

    cout << "Enter rows and columns of Matrix A (m n): ";
    cin >> m >> n;

    cout << "Enter columns of Matrix B (p): ";
    cin >> p;

    // A is m x n, B is n x p
    int sizeA = m * n * sizeof(int);
    int sizeB = n * p * sizeof(int);
    int sizeC = m * p * sizeof(int);

    int *A = new int[m * n];
    int *B = new int[n * p];
    int *C = new int[m * p];

    // Input A
    cout << "Enter elements of Matrix A (" << m << "x" << n << "):\n";
    for (int i = 0; i < m; i++) {
        for (int j = 0; j < n; j++) {
            cin >> A[i * n + j];
        }
    }

    // Input B
    cout << "Enter elements of Matrix B (" << n << "x" << p << "):\n";
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < p; j++) {
            cin >> B[i * p + j];
        }
    }

    int *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, sizeA);
    cudaMalloc(&d_B, sizeB);
    cudaMalloc(&d_C, sizeC);

    cudaMemcpy(d_A, A, sizeA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, sizeB, cudaMemcpyHostToDevice);

    dim3 threads(16, 16);
    dim3 blocks((p + 15) / 16, (m + 15) / 16);

    matrixMul<<<blocks, threads>>>(d_A, d_B, d_C, m, n, p);

    cudaMemcpy(C, d_C, sizeC, cudaMemcpyDeviceToHost);

    cout << "\nResult Matrix (" << m << "x" << p << "):\n";
    for (int i = 0; i < m; i++) {
        for (int j = 0; j < p; j++) {
            cout << C[i * p + j] << " ";
        }
        cout << endl;
    }

    delete[] A;
    delete[] B;
    delete[] C;

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing matrix_mul.cu


In [2]:
!nvcc matrix_mul.cu -o matrix_mul


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [6]:
!./matrix_mul

Enter rows and columns of Matrix A (m n): 2 2
Enter columns of Matrix B (p): 2
Enter elements of Matrix A (2x2):
1 2
3 4
Enter elements of Matrix B (2x2):
5 6
7 8

Result Matrix (2x2):
19 22 
43 50 
